### Procesamiento de Lenguaje Natural I
# **Desafío 1**



### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score, accuracy_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [2]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [3]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [4]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [5]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [6]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [7]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [8]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [9]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [10]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [11]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [12]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [13]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [14]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [15]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [16]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [17]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [18]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [19]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [20]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [21]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [22]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Resolucion

### 1. Vectorizar documentos

In [23]:
indexes = np.random.choice(len(newsgroups_train.data), size=5, replace=False)
print(indexes)

[ 3410  1471 11076  7285  2663]


In [24]:
import pandas as pd

def train_doc_label(doc_index):
    return newsgroups_train.target_names[y_train[doc_index]]

def evaluate_similarity(doc_index, sample=5, text_size=500):
    cossim = cosine_similarity(X_train[doc_index], X_train)[0]
    mostsim_indices = np.argsort(cossim)[::-1][1:sample+1]
    mostsim_scores = cossim[mostsim_indices]
    mostsim_labels = [train_doc_label(i) for i in mostsim_indices]
    
    df_similar = pd.DataFrame({
        'Document Index': mostsim_indices,
        'Cosine Similarity': mostsim_scores,
        'Target Label': mostsim_labels
    })
    
    display(df_similar)

    pd.set_option('display.max_colwidth', None)

    best = mostsim_indices[0]
    text_df = pd.DataFrame({
        "Target Text": [newsgroups_train.data[doc_index][:text_size]],
        "Best Match Text": [newsgroups_train.data[best][:text_size]]
    })

    display(text_df)
    
    pd.reset_option('display.max_colwidth')

In [25]:
for n, idx in enumerate(indexes, start=1):
    print(f"{n}) Target Doc Index:", idx, "Label:", train_doc_label(idx))
    evaluate_similarity(idx)

1) Target Doc Index: 3410 Label: sci.crypt


,Document Index,Cosine Similarity,Target Label
0,2350,0.698786,sci.crypt
1,4682,0.524389,sci.crypt
2,10836,0.491057,alt.atheism
3,6894,0.477847,talk.politics.guns
4,913,0.476579,alt.atheism


,Target Text,Best Match Text
0,Archive-name: net-privacy/part3\nLast-modified: 1993/3/3\nVersion: 2.1\n\n\nNOTES on ANONYMITY on the INTERNET\n==================================\n\nCompiled by L. Detweiler <ld231782@longs.lance.colostate.edu>.\n\n\n<8.1> What are some known anonymous remailing and posting sites?\n<8.2> What are the responsibilities associated with anonymity?\n<8.3> How do I `kill' anonymous postings?\n<8.4> What is the history behind anonymous posting servers?\n<8.5> What is the value of anonymity?\n<8.6> Should anonymous po,"Archive-name: net-privacy/part1\nLast-modified: 1993/3/3\nVersion: 2.1\n\n\nIDENTITY, PRIVACY, and ANONYMITY on the INTERNET\n================================================\n\n(c) 1993 L. Detweiler. Not for commercial use except by permission\nfrom author, otherwise may be freely copied. Not to be altered. \nPlease credit if quoted.\n\nSUMMARY\n=======\n\nInformation on email and account privacy, anonymous mailing and \nposting, encryption, and other privacy and rights issues associated\nwith use of the Inte"


2) Target Doc Index: 1471 Label: comp.graphics


,Document Index,Cosine Similarity,Target Label
0,3370,0.327448,comp.graphics
1,7018,0.325989,comp.graphics
2,2844,0.310871,comp.graphics
3,5091,0.305606,comp.graphics
4,2144,0.301571,comp.graphics


,Target Text,Best Match Text
0,"Well, the temp file thing creates an obvious problem: it is impossible to use\ncview for viewing CD-ROM based picture collections. And it is the ONLY non-\nwindows viewer that works properly with my Cirrus-based 24 bit VGA.\n\n","A very kind soul has mailed me this reply for the bugs in CView.\nSince he isn't in the position to post this himself, he asked me to post\nit for him, but to leave his name out. So here it comes:\n\nCView has quite a number of bugs. The one you mention is perhaps the most\nannoying, but not the most dangerous. As far as I can determine, it has to\ndo with the temp files that CView creates. CView gives the user no control\nover where it places its temp files: it just places them in its\n""current dire"


... (5 outputs left out)

,Target Text,Best Match Text
0,Hmmm. These don't look like references to me. Is passive-aggressive\nbehavior associated with weight rebound? :-),"\nAs mentioned in Adiposity 101, only some experience weight\nrebound. The fact that you don't doesn't prove it doesn't\nhappen to others."


5) Target Doc Index: 2663 Label: rec.autos


,Document Index,Cosine Similarity,Target Label
0,9580,0.315505,rec.autos
1,2086,0.208809,rec.autos
2,2554,0.208169,rec.autos
3,571,0.199867,talk.politics.mideast
4,6224,0.198972,rec.autos


,Target Text,Best Match Text
0,"\nAny advice on HOW to buy a Land Cruiser? My local Toyota dealer says they\nget two a year, and if I want one I can just get on the waiting list.\nForget about a test drive or even kicking the tires.\nAnd if they are that rare, I doubt there is much of a parts inventory on\nhand.\n\n\n","hand.\n\nLand Crusier is just simply nice with shit-load of power and room.\nFully stocked, it cost ~$40,000. I think it is worth the money.\nOnly problem is when you get into accident(it doesn't usually break down\nunless you forget to get a oil change for 30000 miles or something\nstupid like this), body parts and other parts are very hard to locate. \nParts are over priced also. One of my friend had accident in past snow\nstorm and he is still waiting for front bumper and passenger side fender.\n\nWe"


### Observaciones:
1. **Coherencia en la clasificación temática:**
   * En la gran mayoría de los casos analizados (por ejemplo, en las categorías de criptografía `sci.crypt`, compras `misc.forsale`, medicina `sci.med` y automóviles `rec.autos`), los 5 documentos más similares pertenecen exactamente a la misma etiqueta o categoría de *20newsgroups* que el documento objetivo. Esto demuestra que la representación TF-IDF combinada con la similaridad coseno es capaz de capturar eficazmente el tema general de un texto sin necesidad de supervisión previa.

2. **Limitaciones del modelo:**
   * Se observan casos puntuales donde se filtran documentos de otras categorías (como ocurrió en el documento de automóviles con un texto de `talk.politics.mideast`). Esto sucede porque la distancia basada puramente en conteos de palabras y TF-IDF es sensible a la longitud de los textos y a la presencia de palabras funcionales compartidas o términos polisémicos, lo que puede generar falsos vecinos cercanos cuando dos textos comparten vocabulario común a pesar de tratar temas distintos.

### 2. Construir un modelo de clasificación por prototipos (tipo zero-shot).

In [26]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline

class ZeroShotClassifier(BaseEstimator, ClassifierMixin):      
    def fit(self, X, y):
        self.X_train_ = X
        self.y_train_ = np.asarray(y)
        self.classes_ = np.unique(y) # Requerido por scikit-learn para clasificadores
        return self
        
    def predict(self, X):
        cossim_matrix = cosine_similarity(X, self.X_train_)
        best_match_indices = np.argmax(cossim_matrix, axis=1)
        return self.y_train_[best_match_indices]

zero_shot = Pipeline([
    ('vectorizer', TfidfVectorizer()), # mismo vectorizador con parametros por defecto
    ('model', ZeroShotClassifier())  # modelo template para definir el pipeline.
])

zero_shot.fit(newsgroups_train.data, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vectorizer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


In [27]:
def evaluate_model(model, model_name):
    preds = model.predict(newsgroups_test.data)
    f1 = f1_score(y_test, preds, average='macro')
    accuracy = accuracy_score(y_test, preds)
    
    display(model)
    print(f"{model_name} Macro F1-Score: {f1:.4f}")
    print(f"{model_name} Accuracy: {accuracy:.4f}")

evaluate_model(zero_shot, "Zero Shot")

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vectorizer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


Zero Shot Macro F1-Score: 0.5050
Zero Shot Accuracy: 0.5089


Aca se observa que el modelo zero-shot tiene sus limitaciones. El modelo acierta la clasificacion de un documento alrededor del 50% de las veces, lo cual podria interpretarse como no lo suficientemente bueno para la mayoria de sus aplicaciones debido al su elevado margen de error.

De todas formas, a pesar de su simplicidad, su desempeno es destacable. Ya que supera ampliamente a un modelo completamente aleatorio (random guessing). Considerando que existen 20 posibles clases, elegir una al azar solo aciertaria un 1/20 ~ 5% de las veces. Este modelo sencillo zero-shot supera este valor 10 veces.

### 3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación

In [28]:
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('model', None)  # modelo template para definir el pipeline.
])

param_grid = [
    {
        'vectorizer__max_df': [0.75, 0.85, 1.0],
        'vectorizer__min_df': [1, 2],
        'vectorizer__sublinear_tf': [True, False],
        'model': [MultinomialNB()],
        'model__alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
    },
    {
        'vectorizer__max_df': [0.75, 0.85, 1.0],
        'vectorizer__min_df': [1, 2],
        'vectorizer__sublinear_tf': [True, False],
        'model': [ComplementNB()],
        'model__alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
    }
]

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1
)

grid_search.fit(newsgroups_train.data, y_train)

print(f"Mejor F1-Score Macro: {grid_search.best_score_:.4f}")
print(f"Mejores parámetros:\n {grid_search.best_params_}")

best_naive_bayes = grid_search.best_estimator_

Mejor F1-Score Macro: 0.7533
Mejores parámetros:
 {'model': ComplementNB(), 'model__alpha': 0.5, 'vectorizer__max_df': 0.85, 'vectorizer__min_df': 1, 'vectorizer__sublinear_tf': True}


In [29]:
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('model', ZeroShotClassifier())  # modelo template para definir el pipeline.
])

param_grid = {
    'vectorizer__max_df': [0.75, 0.85, 1.0],
    'vectorizer__min_df': [1, 2],
    'vectorizer__sublinear_tf': [True, False]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1
)

grid_search.fit(newsgroups_train.data, y_train)

print(f"Mejor F1-Score Macro: {grid_search.best_score_:.4f}")
print(f"Mejores parámetros:\n {grid_search.best_params_}")

zero_shot_optimized = grid_search.best_estimator_

Mejor F1-Score Macro: 0.6754
Mejores parámetros:
 {'vectorizer__max_df': 0.75, 'vectorizer__min_df': 1, 'vectorizer__sublinear_tf': True}


In [30]:
evaluate_model(best_naive_bayes, "Naive Bayes")
evaluate_model(zero_shot_optimized, "Zero Shot (optimized)")

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vectorizer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.85
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


Naive Bayes Macro F1-Score: 0.6968
Naive Bayes Accuracy: 0.7173


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vectorizer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.75
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


Zero Shot (optimized) Macro F1-Score: 0.5672
Zero Shot (optimized) Accuracy: 0.5730


Inicialmente, el primer modelo de Naive Bayes obtuvo un F1-Score de 0.58, apenas por encima del logrado por el modelo zero-shot (0.50) utilizando el mismo vectorizador con parámetros por defecto.

Al optimizar los hiperparámetros del vectorizador de forma independiente para cada enfoque y ajustar los del clasificador, se observa que Naive Bayes (ComplementNB) resulta considerablemente superior al modelo zero-shot.

Si bien optimizar los parámetros del vectorizador del modelo zero-shot mejora su desempeño alrededor de un 7% (alcanzando un F1 cercano al 0.57, casi igualando al Naive Bayes original sin tunear de 0.58), el modelo Naive Bayes con sus parámetros optimizados mejora considerablemente, superando el 70% de precisión.

#### Observaciones:
Escalamiento de frecuencias con sublinear_tf=True: Que este parámetro haya resultado óptimo significa que la transformación logarítmica de la frecuencia de los términos superó a la frecuencia bruta lineal. Esto se interpreta teóricamente en que la importancia de una palabra en un documento no crece de forma lineal (es decir, que aparezca 10 veces no la hace 10 veces más importante que aparecer 1 vez), mitigando el sesgo de documentos excesivamente largos y dándole un peso más equilibrado al vocabulario general.

Filtros de frecuencia (max_df y min_df): Al descartar palabras extremadamente frecuentes (max_df restrictivo) y términos únicos o erratas de una sola aparición (min_df=1 o 2), el modelo elimina ruido estocástico del espacio vectorial. Esto reduce la dimensionalidad de las matrices dispersas (sparse matrices), permitiendo que ComplementNB estime con mayor robustez las probabilidades condicionales por clase sin sobreajustarse a vocablos irrelevantes (como palabras funcionales o conjunciones mal filtradas).

### 4. Transponer la matriz documento-término

In [31]:
palabras_vec = X_train.T

palabras_elegidas = ["computer", "car", "president", "god", "ball"]
indices_palabras = [tfidfvect.vocabulary_[p] for p in palabras_elegidas]

for idx, palabra in zip(indices_palabras, palabras_elegidas):
    sim_palabras = cosine_similarity(palabras_vec[idx], palabras_vec)[0]
    
    # Indices de las 5 palabras más similares
    top_5_similares = np.argsort(sim_palabras)[::-1][1:6]
    
    print(f"\nPalabras más similares a: '{palabra}'")
    
    datos = []
    for sim_idx in top_5_similares:
        palabra_similar = idx2word[sim_idx]
        score = sim_palabras[sim_idx]
        datos.append({"word": palabra_similar, "score": score})
    
    df_similares = pd.DataFrame(datos)
    display(df_similares)


Palabras más similares a: 'computer'


,word,score
0,decwriter,0.156274
1,deluged,0.152190
2,harkens,0.152190
3,shopper,0.144323
4,the,0.136142



Palabras más similares a: 'car'


,word,score
0,cars,0.179666
1,criterium,0.177040
2,civic,0.174773
3,owner,0.168906
4,dealer,0.168111



Palabras más similares a: 'president'


,word,score
0,miyazawa,0.421564
1,______________________________________________...,0.355431
2,oval,0.296477
3,mubarek,0.289955
4,secretary,0.285429



Palabras más similares a: 'god'


,word,score
0,jesus,0.268789
1,bible,0.261616
2,that,0.256032
3,existence,0.254772
4,christ,0.251052



Palabras más similares a: 'ball'


,word,score
0,runners,0.321696
1,relieves,0.303031
2,puzzled,0.293456
3,runner,0.282936
4,swing,0.279594


#### Observaciones:
1. **Casos con alto sentido semántico y temático:**
   * **`"car"`:** Arrojó resultados sumamente coherentes dentro del dominio automotriz, destacando términos como `"cars"`, `"civic"`, `"owner"` y `"dealer"`. Esto demuestra que las palabras que comparten contextos documentales frecuentes agrupan de forma natural entidades y conceptos relacionados.
   * **`"god"`:** Presenta una fuerte coherencia semántica vinculada a discusiones religiosas, encontrando términos muy cercanos como `"jesus"`, `"bible"` y `"christ"`.
   * **`"president"`:** Muestra una clara asociación con el ámbito político e institucional, destacando términos como `"oval"` (en referencia al despacho oval) y `"secretary"`, además de nombres propios del contexto informativo de la época como `"miyazawa"` (primer ministro de Japón entre noviembre de 1991 y agosto de 1993).

2. **Casos con menor interpretabilidad o ruido:**
   * **`"computer"` y `"ball"`:** Si bien algunos resultados guardan relación contextual (como `"decwriter"` para computación o `"runners"` y `"runner"` para el ámbito deportivo), también aparecen términos inesperados o de baja relevancia semántica (como `"deluged"`, `"harkens"` o `"relieves"`). 